# Entraînement avec augmentation de luminosité et de contraste — v2 (CBAM résiduel, configurations propres)

**Question :** exposer les modèles à des variations d’intensité pendant l’apprentissage peut-il réduire la sensibilité observée aux images plus sombres ?

**Ce qui change par rapport à la v1** : les points de départ sont ceux des six modèles du notebook `07_stabilite_multigraines_v2`. Le CBAM inclut la connexion résiduelle et chaque architecture garde **sa propre configuration retenue** (classique : LR 0,001 et Dropout 0,5 ; CBAM résiduel : LR 0,0005 et Dropout 0,3). La recette d’augmentation, le budget et le critère d’arrêt sont inchangés. L’identifiant de la recette (`luminosite_contraste_v1`) est volontairement conservé : il sert de clé au générateur, donc chaque image reçoit exactement les mêmes facteurs de luminosité et de contraste que dans la conception initiale.

Ce notebook entraîne le CNN classique et le CNN avec CBAM pour les graines **42, 123 et 2026**. Chaque essai repart de sa sauvegarde `epoque_000.pt`, créée avant l’entraînement sans augmentation. Il conserve les données, l’initialisation du modèle, l’état initial de l’optimiseur, l’ordre initial des lots et l’état aléatoire PyTorch de cet essai.

| Élément | Choix fixé pour cette expérience |
|---|---|
| Optimiseur | Adam, weight decay 0 ; taux d’apprentissage 0,001 (classique) ou 0,0005 (CBAM) |
| Dropout | 0,5 (classique) ou 0,3 (CBAM) |
| Lot | 16 images, calcul sur CPU |
| Budget | Maximum 20 époques ; patience 4 sans amélioration stricte du F1 macro de validation |
| Luminosité d’apprentissage | Facteur uniforme entre 0,90 et 1,10 |
| Contraste d’apprentissage | Facteur uniforme entre 0,90 et 1,10 |
| Ordre des transformations | Aléatoire : luminosité puis contraste, ou l’inverse |
| Zone transformée | Rectangle de l’image redimensionnée ; padding ajouté conservé |
| Validation | Préparation habituelle, sans augmentation aléatoire |

Les deux transformations sont appliquées à chaque image d’apprentissage. Leurs facteurs changent à chaque époque. Un générateur séparé, indexé par **graine, époque et chemin d’image**, garantit les mêmes transformations pour les deux architectures à graine et époque égales. Il ne consomme pas les tirages PyTorch utilisés par le dropout et le chargement des lots.

Les images originales et les anciennes expériences restent disponibles. Aucun ensemble de test n’est chargé. Aucun seuil de décision n’est modifié. Les six nouvelles expériences sont sauvegardées dans `optimisation/augmentation_luminosite_contraste_v2` ; la v1 n’est pas modifiée.

**Exécution :** ouvrir avec le noyau `memoire_cnn`, puis exécuter les cellules dans l’ordre. Une cellule affiche les augmentations sur deux images d’apprentissage. La cellule « Lancer les six entraînements » peut prendre plusieurs heures. Après une interruption, réexécuter ce notebook : les essais terminés sont reconnus et les autres reprennent après la dernière époque complète. Ne pas exécuter deux copies simultanément.

**Limite méthodologique :** ce protocole a été choisi après observation de la robustesse sur validation. Il s’agit d’une nouvelle exploration, avec trois graines seulement. La graine 42 avait également servi à sélectionner les hyperparamètres. Le test avait déjà été consulté plus tôt dans le projet. Ces résultats ne constituent pas une validation indépendante ou clinique. Autre limite : la plage d’augmentation [0,90 ; 1,10] englobe exactement les niveaux de luminosité et de contraste testés dans le notebook 08. Une amélioration sur ces conditions montrera que le modèle a appris à tolérer des variations qu’il a vues pendant l’entraînement, pas qu’il est robuste à des variations plus fortes.

Ce notebook fournit la comparaison **sur validation originale**. Les nouvelles sauvegardes permettront ensuite de répéter exactement les sept conditions de robustesse du notebook 08. Une hausse du score sur les images originales, seule, ne démontre pas une amélioration de robustesse.


In [ ]:
from pathlib import Path, PureWindowsPath
import hashlib
import json
import shutil
import sys
import time
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn import __version__ as version_sklearn
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from tqdm.auto import tqdm
from IPython.display import display

PROJET = Path(r"C:\Users\David KUNYIMA\OneDrive\Desktop\Memoire_CNN_Attention")
IMAGES = PROJET / "archive" / "chest_xray"
REPARTITION = PROJET / "repartitions" / "repartition_20260917_102959_713272"
FICHIER_REFERENCE = PROJET / "optimisation" / "stabilite_multigraines_v2" / "resultats_multigraines.csv"
RACINE = PROJET / "optimisation" / "augmentation_luminosite_contraste_v2"
APPAREIL = torch.device("cpu")
CLASSES = {"NORMAL": 0, "PNEUMONIA": 1}
GRAINES = [42, 123, 2026]
TAILLE, LOT, MAX_EPOQUES, PATIENCE = 224, 16, 20, 4
# Configuration retenue propre à chaque architecture (voir notebook 07 v2).
CONFIGURATIONS = {
    "CNN_classique": {"indice": 2, "lr": 0.001, "dropout": 0.5},
    "CNN_CBAM": {"indice": 3, "lr": 0.0005, "dropout": 0.3},
}
PRETRAITEMENT = "L; ImageOps.pad 224 bilineaire noir centre; /255"
AUGMENTATION = {
    "version": "luminosite_contraste_v1",
    "luminosite": [0.90, 1.10], "contraste": [0.90, 1.10],
    "probabilite": 1.0, "ordre": "aleatoire entre les deux ordres possibles",
    "zone": "rectangle image redimensionnee, padding ajoute preserve",
    "contraste_centre": "moyenne float64 du rectangle au moment de l’operation",
    "ecretage": "[0,1] apres chaque operation; pixels float32",
    "generateur": "numpy PCG64, SHA256(version|graine|epoque|chemin), 8 premiers octets little-endian",
}
VERSIONS = {
    "python": sys.version, "torch": str(torch.__version__), "numpy": np.__version__,
    "pandas": pd.__version__, "pillow": PIL.__version__, "sklearn": version_sklearn,
}
torch.use_deterministic_algorithms(True)

def exiger(condition, message):
    if not condition:
        raise RuntimeError(message)

def sha256_fichier(chemin):
    h = hashlib.sha256()
    with open(chemin, "rb") as fichier:
        for bloc in iter(lambda: fichier.read(1024 * 1024), b""):
            h.update(bloc)
    return h.hexdigest()

def ecrire_json(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    temporaire.write_text(json.dumps(objet, ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
    temporaire.replace(chemin)

def ecrire_csv(tableau, chemin):
    temporaire = chemin.with_suffix(".tmp")
    tableau.to_csv(temporaire, index=False, encoding="utf-8-sig", float_format="%.17g")
    temporaire.replace(chemin)

def sauver_torch(objet, chemin):
    temporaire = chemin.with_suffix(".tmp")
    torch.save(objet, temporaire)
    temporaire.replace(chemin)

print("Appareil :", APPAREIL, "| PyTorch :", torch.__version__)
print("Six essais. Configurations :", CONFIGURATIONS)


## 1. Recharger les données

Les contrôles reprennent ceux des notebooks précédents : effectifs attendus et absence de chevauchement de groupes, d’empreintes et de chemins entre apprentissage et validation. Les groupes reposent sur les identifiants déduits des fichiers ; l’identité clinique des patients n’a pas été vérifiée.


In [ ]:
train_df = pd.read_csv(REPARTITION / "train.csv", dtype={"groupe": str, "empreinte": str})
val_df = pd.read_csv(REPARTITION / "val.csv", dtype={"groupe": str, "empreinte": str})
for nom, tableau, attendus in [("train", train_df, {"NORMAL": 1079, "PNEUMONIA": 2721}), ("val", val_df, {"NORMAL": 269, "PNEUMONIA": 684})]:
    colonnes = ["classe", "groupe", "empreinte", "chemin_relatif"]
    exiger(set(colonnes).issubset(tableau.columns), f"Colonnes absentes de {nom}.")
    exiger(not tableau[colonnes].isna().any().any(), f"Valeurs manquantes dans {nom}.")
    exiger(tableau["classe"].value_counts().to_dict() == attendus, f"Effectifs inattendus : {nom}.")
    exiger(tableau["chemin_relatif"].is_unique, f"Chemins dupliqués : {nom}.")
    absents = [p for p in tableau["chemin_relatif"] if not (IMAGES / p).is_file()]
    exiger(not absents, f"Images absentes dans {nom} : {absents[:3]}")
for colonne in ["groupe", "empreinte", "chemin_relatif"]:
    exiger(set(train_df[colonne]).isdisjoint(val_df[colonne]), f"Chevauchement : {colonne}.")
effectifs = train_df["classe"].value_counts()
POIDS = torch.tensor([len(train_df)/(2*effectifs[c]) for c in CLASSES], dtype=torch.float32)
SHA_CSV = {nom: sha256_fichier(REPARTITION / f"{nom}.csv") for nom in ["train", "val"]}
print("Apprentissage :", len(train_df), "| Validation :", len(val_df))
print("Poids NORMAL / PNEUMONIA :", POIDS.tolist())


## 2. Définir l’augmentation

**Luminosité :** multiplication des intensités. **Contraste :** multiplication des écarts à la moyenne de l’image. Exemple : un facteur de luminosité de 0,94 assombrit l’image. Les pixels sont limités à [0, 1] après chaque opération.

La transformation est appliquée au rectangle de l’image redimensionnée, y compris son fond et ses annotations. Le masque utilisé repère uniquement le padding ajouté ; ce n’est pas une segmentation anatomique. Les fonctions sont explicites pour conserver exactement cette zone.


In [ ]:
def preparer_avec_masque(image):
    gris = image.convert("L")
    preparee = ImageOps.pad(
        gris, (TAILLE, TAILLE), method=Image.Resampling.BILINEAR,
        color=0, centering=(0.5, 0.5)
    )
    # Toute l'image source est marquée, même ses pixels initialement noirs.
    blanc = Image.new("L", gris.size, color=255)
    masque = ImageOps.pad(
        blanc, (TAILLE, TAILLE), method=Image.Resampling.NEAREST,
        color=0, centering=(0.5, 0.5)
    )
    return np.array(preparee, dtype=np.uint8), np.array(masque) > 0


def parametres_augmentation(graine, epoque, chemin_relatif):
    cle = f"{AUGMENTATION['version']}|{graine}|{epoque}|{chemin_relatif}".encode("utf-8")
    graine_locale = int.from_bytes(hashlib.sha256(cle).digest()[:8], "little")
    generateur = np.random.Generator(np.random.PCG64(graine_locale))
    luminosite = float(generateur.uniform(*AUGMENTATION["luminosite"]))
    contraste = float(generateur.uniform(*AUGMENTATION["contraste"]))
    ordre = ["luminosite", "contraste"] if generateur.integers(2) == 0 else ["contraste", "luminosite"]
    return {"luminosite": luminosite, "contraste": contraste, "ordre": ordre}

def appliquer_augmentation(tableau_u8, masque, parametres):
    x = tableau_u8.astype(np.float32) / 255.0
    for operation in parametres["ordre"]:
        if operation == "luminosite":
            x[masque] = x[masque] * np.float32(parametres["luminosite"])
        elif operation == "contraste":
            moyenne = x[masque].mean(dtype=np.float64)
            x[masque] = (x[masque] - moyenne) * parametres["contraste"] + moyenne
        else:
            raise ValueError(f"Opération inconnue : {operation}")
        np.clip(x, 0.0, 1.0, out=x)
    return x

class RadiographiesAugmentees(Dataset):
    def __init__(self, tableau, augmentation=False, graine=42):
        self.tableau = tableau.reset_index(drop=True)
        self.augmentation = augmentation
        self.graine = graine
        self.epoque = 0
    def __len__(self):
        return len(self.tableau)
    def __getitem__(self, indice):
        ligne = self.tableau.iloc[indice]
        with Image.open(IMAGES / ligne["chemin_relatif"]) as image:
            if self.augmentation:
                exiger(self.epoque >= 1, "Définir l’époque avant de charger les images augmentées.")
                preparee, masque = preparer_avec_masque(image)
                parametres = parametres_augmentation(self.graine, self.epoque, ligne["chemin_relatif"])
                x = appliquer_augmentation(preparee, masque, parametres)
            else:
                # Préparation de validation identique au protocole d’origine.
                preparee = ImageOps.pad(image.convert("L"), (TAILLE, TAILLE), method=Image.Resampling.BILINEAR, color=0, centering=(0.5, 0.5))
                x = np.array(preparee, dtype=np.float32) / 255.0
        return torch.from_numpy(x).unsqueeze(0), CLASSES[ligne["classe"]]

VAL = RadiographiesAugmentees(val_df, augmentation=False)
print("Augmentation définie uniquement pour l’apprentissage.")


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ligne, classe in enumerate(CLASSES):
    exemple = train_df[train_df["classe"] == classe].iloc[0]
    with Image.open(IMAGES / exemple["chemin_relatif"]) as image:
        preparee, masque = preparer_avec_masque(image)
    axes[ligne, 0].imshow(preparee, cmap="gray", vmin=0, vmax=255)
    axes[ligne, 0].set_title(f"{classe} — originale")
    for colonne, epoque in enumerate([1, 2, 3], start=1):
        parametres = parametres_augmentation(42, epoque, exemple["chemin_relatif"])
        x = appliquer_augmentation(preparee, masque, parametres)
        exiger(np.isfinite(x).all() and 0 <= x.min() <= x.max() <= 1, "Pixels incorrects.")
        exiger((x[~masque] == 0).all(), "Padding modifié.")
        axes[ligne, colonne].imshow(x, cmap="gray", vmin=0, vmax=1)
        axes[ligne, colonne].set_title(f"Époque {epoque}\nL={parametres['luminosite']:.3f} ; C={parametres['contraste']:.3f}", fontsize=10)
    for axe in axes[ligne]:
        axe.axis("off")
fig.suptitle("Exemples d’apprentissage, graine 42 — L : luminosité ; C : contraste")
fig.tight_layout()
plt.show()
figure_exemples = fig


## 3. Retrouver les points de départ des essais précédents

Les sauvegardes `epoque_000.pt` contiennent les modèles **avant tout apprentissage**. Les six nouvelles expériences repartent de ces mêmes points de départ. L’ancienne sauvegarde `meilleur.pt` est utilisée uniquement pour vérifier la référence de comparaison.

Si un fichier manque, retrouver le dossier de l’expérience concernée. Le notebook ne remplacera pas silencieusement une référence par un autre modèle.


In [ ]:
class CNNClassique(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bloc1 = nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc2 = nn.Sequential(nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.bloc3 = nn.Sequential(nn.Conv2d(32,64,3,padding=1),nn.ReLU(),nn.MaxPool2d(2))
        self.reduction = nn.AdaptiveAvgPool2d((4,4))
        self.classificateur = nn.Sequential(nn.Flatten(),nn.Linear(64*4*4,64),nn.ReLU(),nn.Dropout(dropout),nn.Linear(64,2))
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(x))

class AttentionCanaux(nn.Module):
    def __init__(self, canaux, reduction=16):
        super().__init__()
        dim=max(1,canaux//reduction)
        self.reseau_partage=nn.Sequential(
            nn.Conv2d(canaux,dim,1,bias=False),nn.ReLU(),
            nn.Conv2d(dim,canaux,1,bias=False))
    def forward(self,x):
        moyenne=x.mean(dim=(2,3),keepdim=True)
        maximum=x.amax(dim=(2,3),keepdim=True)
        return x*torch.sigmoid(self.reseau_partage(moyenne)+self.reseau_partage(maximum))

class AttentionSpatiale(nn.Module):
    def __init__(self):
        super().__init__()
        self.convolution=nn.Conv2d(2,1,7,padding=3,bias=False)
    def forward(self,x):
        resume=torch.cat([x.mean(dim=1,keepdim=True),x.amax(dim=1,keepdim=True)],dim=1)
        return x*torch.sigmoid(self.convolution(resume))

class CBAM(nn.Module):
    def __init__(self,canaux=64,reduction=16):
        super().__init__()
        self.attention_canaux=AttentionCanaux(canaux,reduction)
        self.attention_spatiale=AttentionSpatiale()
    def forward(self,x):
        # v2 : connexion résiduelle, identique aux notebooks 06 v2, 07 v2 et 08 v2.
        attenue = self.attention_spatiale(self.attention_canaux(x))
        return x + attenue

class CNNAvecCBAM(CNNClassique):
    def __init__(self,dropout=0.3):
        super().__init__(dropout)
        self.cbam=CBAM(64,16)
    def forward(self,x):
        x=self.bloc3(self.bloc2(self.bloc1(x)))
        return self.classificateur(self.reduction(self.cbam(x)))


In [ ]:
ARCHITECTURES = {
    "CNN_classique": (CNNClassique, "CNNClassique_89026_v1", 89026),
    "CNN_CBAM": (CNNAvecCBAM, "CNNAvecCBAM_89636_apres_bloc3_r16_k7_residuel_v2", 89636),
}
BASES = pd.read_csv(FICHIER_REFERENCE)
obligatoires = {"modele", "graine", "lr", "dropout", "epoque_retenue", "sauvegarde", "val_f1_macro", "val_exactitude", "val_rappel_pneumonie", "val_specificite_NORMAL"}
exiger(obligatoires.issubset(BASES.columns), "Tableau de référence incomplet.")
exiger(not BASES[list(obligatoires)].isna().any().any(), "Valeurs manquantes dans les références.")
exiger(len(BASES) == 6 and not BASES.duplicated(["modele", "graine"]).any(), "Il faut six références distinctes.")
exiger(set(zip(BASES["modele"], BASES["graine"])) == {(m, g) for m in ARCHITECTURES for g in GRAINES}, "Références inattendues.")
for nom_config, reglages in CONFIGURATIONS.items():
    sous_table = BASES[BASES["modele"] == nom_config]
    exiger(np.allclose(sous_table["lr"], reglages["lr"], rtol=0, atol=1e-12)
           and np.allclose(sous_table["dropout"], reglages["dropout"], rtol=0, atol=1e-12),
           f"Réglages différents de ceux retenus pour {nom_config}.")

def chemin_reference(texte):
    parties = PureWindowsPath(str(texte)).parts
    positions = [i for i, partie in enumerate(parties) if partie == "optimisation"]
    exiger(len(positions) == 1, f"Chemin de référence inattendu : {texte}")
    morceaux = parties[positions[0]:]
    exiger(".." not in morceaux, "Chemin relatif incorrect.")
    return PROJET.joinpath(*morceaux)

ORIGINES, ETATS_INITIAUX = {}, {}
for reference in BASES.to_dict("records"):
    nom, graine = reference["modele"], int(reference["graine"])
    classe, architecture, nombre_parametres = ARCHITECTURES[nom]
    meilleur = chemin_reference(reference["sauvegarde"])
    initial = meilleur.parent / "epoque_000.pt"
    exiger(initial.is_file() and meilleur.is_file(), f"Sauvegarde manquante : {initial} ou {meilleur}")
    etat = torch.load(initial, map_location="cpu", weights_only=True)
    protocole = etat["protocole"]
    exiger(etat["epoque"] == 0 and not etat["historique"], "Le point de départ a déjà été entraîné.")
    exiger(not etat["optimiseur"]["state"], "L’optimiseur initial possède déjà un historique de mise à jour.")
    exiger(etat["configuration"] == CONFIGURATIONS[nom], "Configuration initiale différente.")
    attendus = {
        "architecture": architecture, "graine": graine, "lot": LOT,
        "max_epoques": MAX_EPOQUES, "patience": PATIENCE,
        "critere": "val_f1_macro", "amelioration": "strictement superieure",
        "optimiseur": "Adam", "weight_decay": 0.0, "classes": CLASSES,
        "poids_classes": POIDS.tolist(), "sha256_csv": SHA_CSV,
        "pretraitement": PRETRAITEMENT + "; aucune augmentation",
    }
    for cle, valeur in attendus.items():
        exiger(protocole.get(cle) == valeur, f"{nom}, graine {graine} : protocole différent pour {cle}.")
    exiger(protocole["versions"] == VERSIONS and protocole["threads_torch"] == torch.get_num_threads(),
           f"{nom}, graine {graine} : environnement différent. Utiliser le noyau et les versions des essais de référence avant de poursuivre.")
    with torch.random.fork_rng(devices=[]):
        verification = classe(CONFIGURATIONS[nom]["dropout"])
        verification.load_state_dict(etat["modele"], strict=True)
        exiger(sum(p.numel() for p in verification.parameters()) == nombre_parametres, "Architecture inattendue.")
    sauvegarde_meilleure = torch.load(meilleur, map_location="cpu", weights_only=True)
    exiger(sauvegarde_meilleure["protocole"] == protocole and sauvegarde_meilleure["configuration"] == etat["configuration"], "Référence et initialisation incompatibles.")
    exiger(sauvegarde_meilleure["epoque"] == int(reference["epoque_retenue"]), "Époque de référence différente.")
    selection = max(sauvegarde_meilleure["historique"], key=lambda r: r["val_f1_macro"])
    exiger(selection["epoque"] == sauvegarde_meilleure["epoque"], "La référence n’est pas la meilleure époque sélectionnée.")
    for metrique in ["val_f1_macro", "val_exactitude", "val_rappel_pneumonie", "val_specificite_NORMAL"]:
        exiger(abs(float(selection[metrique]) - float(reference[metrique])) < 1e-10, "Le tableau et la sauvegarde de référence diffèrent.")
    cle = (nom, graine)
    ETATS_INITIAUX[cle] = etat
    ORIGINES[cle] = {
        "modele": nom, "graine": graine,
        "initialisation_relative": initial.relative_to(PROJET).as_posix(),
        "sha256_initialisation": sha256_fichier(initial),
        "reference_relative": meilleur.relative_to(PROJET).as_posix(),
        "sha256_reference": sha256_fichier(meilleur),
    }
    del verification, sauvegarde_meilleure

PROTOCOLE_COMMUN = {
    "version": "augmentation_luminosite_contraste_v2", "graines": GRAINES,
    "configurations": CONFIGURATIONS, "lot": LOT, "max_epoques": MAX_EPOQUES, "patience": PATIENCE,
    "optimiseur": "Adam", "weight_decay": 0.0,
    "critere": "val_f1_macro", "amelioration": "strictement superieure",
    "classes": CLASSES, "poids_classes": POIDS.tolist(), "sha256_csv": SHA_CSV,
    "sha256_tableau_reference": sha256_fichier(FICHIER_REFERENCE),
    "pretraitement": PRETRAITEMENT, "augmentation_train": AUGMENTATION,
    "augmentation_validation": False, "appareil": str(APPAREIL),
    "versions": VERSIONS, "threads_torch": torch.get_num_threads(),
    "origines": [ORIGINES[(nom, graine)] for nom in ARCHITECTURES for graine in GRAINES],
    "limite": "Augmentation choisie apres observation de la robustesse sur validation; test deja consulte; analyse exploratoire; plage d augmentation identique aux niveaux de robustesse evalues; CBAM residuel v2; configuration propre a chaque architecture.",
}
RACINE.mkdir(parents=True, exist_ok=True)
fichier_protocole = RACINE / "protocole.json"
if fichier_protocole.exists():
    exiger(json.loads(fichier_protocole.read_text(encoding="utf-8")) == PROTOCOLE_COMMUN,
           "Le protocole a changé. Conserver les résultats existants et utiliser un autre nom RACINE pour une nouvelle expérience.")
else:
    ecrire_json(PROTOCOLE_COMMUN, fichier_protocole)
figure_exemples.savefig(RACINE / "exemples_augmentation_train.png", dpi=160, bbox_inches="tight")
plt.close(figure_exemples)
display(BASES[["modele", "graine", "epoque_retenue", "val_f1_macro"]].round(5))
print("Six points de départ vérifiés. Aucun nouvel entraînement lancé jusqu’ici.")


## 4. Fonctions d’entraînement et de reprise

Le calcul des mesures et de la perte pondérée est repris du notebook 07. Une sauvegarde est créée après chaque époque complète, avec les paramètres, l’optimiseur et les états aléatoires.

L’augmentation est déterminée par le numéro d’époque et le chemin de l’image. Elle reste donc identique si une époque interrompue doit être recommencée. La meilleure époque est toujours choisie sur le F1 macro de la validation **originale**.

Les mesures d’apprentissage sont calculées sur les images augmentées au fil des mises à jour, avec dropout actif. Elles ne doivent pas être comparées directement aux anciennes mesures d’apprentissage sur images non augmentées pour conclure sur l’effet de l’augmentation.


In [ ]:
def une_epoque(modele, chargeur, critere, optimiseur=None):
    apprentissage = optimiseur is not None
    modele.train(apprentissage)
    pertes, poids_total = 0.0, 0.0
    reels, predits = [], []
    for images, etiquettes in tqdm(chargeur, desc="Apprentissage" if apprentissage else "Validation", leave=False):
        images, etiquettes = images.to(APPAREIL), etiquettes.to(APPAREIL)
        if apprentissage: optimiseur.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(apprentissage):
            scores = modele(images)
            perte = critere(scores, etiquettes)
            if not torch.isfinite(perte).item(): raise RuntimeError("Perte non finie : arrêter et examiner cette configuration.")
            if apprentissage:
                perte.backward()
                optimiseur.step()
        poids = critere.weight[etiquettes].sum().item()
        pertes += perte.item()*poids
        poids_total += poids
        reels.extend(etiquettes.cpu().tolist())
        predits.extend(scores.detach().argmax(1).cpu().tolist())
    cm = confusion_matrix(reels, predits, labels=[0,1])
    tn,fp,fn,tp = cm.ravel()
    mesures = {
        "perte": float(pertes/poids_total), "exactitude": float(accuracy_score(reels,predits)),
        "precision_pneumonie": float(precision_score(reels,predits,zero_division=0)),
        "rappel_pneumonie": float(recall_score(reels,predits,zero_division=0)),
        "specificite_NORMAL": float(tn/(tn+fp)),
        "f1_pneumonie": float(f1_score(reels,predits,zero_division=0)),
        "f1_macro": float(f1_score(reels,predits,labels=[0,1],average="macro",zero_division=0))
    }
    return mesures, cm.tolist()


def choisir_meilleure(historique):
    # max retient la première occurrence en cas d’égalité, comme auparavant.
    return max(historique, key=lambda r: r["val_f1_macro"])

def patience_atteinte(historique):
    return bool(historique) and len(historique) - choisir_meilleure(historique)["epoque"] >= PATIENCE

def entrainer_avec_augmentation(nom, graine):
    classe, architecture, _ = ARCHITECTURES[nom]
    configuration = dict(CONFIGURATIONS[nom])
    dossier = RACINE / nom / f"graine_{graine}" / f"configuration_{configuration['indice']:02d}"
    dossier.mkdir(parents=True, exist_ok=True)
    protocole = {**PROTOCOLE_COMMUN, "modele": nom, "architecture": architecture,
                 "graine": graine, "origine_essai": ORIGINES[(nom, graine)]}
    modele = classe(configuration["dropout"]).to(APPAREIL)
    optimiseur = torch.optim.Adam(modele.parameters(), lr=configuration["lr"])
    critere = nn.CrossEntropyLoss(weight=POIDS.to(APPAREIL))
    donnees_train = RadiographiesAugmentees(train_df, augmentation=True, graine=graine)
    generateur = torch.Generator()
    chargeur_train = DataLoader(donnees_train, batch_size=LOT, shuffle=True, generator=generateur, num_workers=0, drop_last=False)
    chargeur_val = DataLoader(VAL, batch_size=LOT, shuffle=False, num_workers=0, drop_last=False)
    disponibles = sorted(dossier.glob("epoque_[0-9][0-9][0-9].pt"))
    if disponibles:
        etat = torch.load(disponibles[-1], map_location="cpu", weights_only=True)
        exiger(etat["configuration"] == configuration and etat["protocole"] == protocole, "Sauvegarde de reprise incompatible.")
        historique = etat["historique"]
        exiger(etat["epoque"] == len(historique), "Historique de reprise incomplet.")
        exiger([r["epoque"] for r in historique] == list(range(1, len(historique)+1)), "Époques de reprise incohérentes.")
        print("Reprise après l’époque", etat["epoque"])
    else:
        etat = ETATS_INITIAUX[(nom, graine)]
        historique = []
        print("Départ depuis l’initialisation de référence.")
    modele.load_state_dict(etat["modele"], strict=True)
    optimiseur.load_state_dict(etat["optimiseur"])
    generateur.set_state(etat["etat_generateur"])
    torch.set_rng_state(etat["etat_torch"])

    def sauvegarder(epoque, matrice=None):
        sauver_torch({
            "epoque": epoque, "modele": modele.state_dict(), "optimiseur": optimiseur.state_dict(),
            "historique": historique, "configuration": configuration, "protocole": protocole,
            "etat_torch": torch.get_rng_state(), "etat_generateur": generateur.get_state(),
            "matrice_validation": matrice,
        }, dossier / f"epoque_{epoque:03d}.pt")

    if not disponibles:
        sauvegarder(0)
    while len(historique) < MAX_EPOQUES and not patience_atteinte(historique):
        epoque = len(historique) + 1
        donnees_train.epoque = epoque
        debut = time.perf_counter()
        mesures_train, _ = une_epoque(modele, chargeur_train, critere, optimiseur)
        mesures_val, matrice = une_epoque(modele, chargeur_val, critere)
        ligne = {"epoque": epoque, "duree_secondes": float(time.perf_counter() - debut)}
        ligne.update({f"train_{k}": v for k, v in mesures_train.items()})
        ligne.update({f"val_{k}": v for k, v in mesures_val.items()})
        historique.append(ligne)
        sauvegarder(epoque, matrice)
        ecrire_csv(pd.DataFrame(historique), dossier / "historique.csv")
        print(f"{nom} | graine {graine} | époque {epoque:02d} | F1 macro val {mesures_val['f1_macro']:.5f} | rappel {mesures_val['rappel_pneumonie']:.4f} | spécificité {mesures_val['specificite_NORMAL']:.4f}")
    meilleur = choisir_meilleure(historique)
    source = dossier / f"epoque_{meilleur['epoque']:03d}.pt"
    temporaire = dossier / "meilleur.tmp"
    shutil.copy2(source, temporaire)
    temporaire.replace(dossier / "meilleur.pt")
    ecrire_csv(pd.DataFrame(historique), dossier / "historique.csv")
    resume = {
        "modele": nom, "graine": graine, "augmentation": AUGMENTATION["version"], **configuration,
        "epoque_retenue": meilleur["epoque"], "epoque_finale": len(historique),
        "arret": "patience" if patience_atteinte(historique) else "maximum_epoques",
        "duree_minutes": sum(r["duree_secondes"] for r in historique)/60,
        "sauvegarde": str(dossier / "meilleur.pt"),
        **{k: v for k, v in meilleur.items() if k.startswith("val_")},
    }
    ecrire_json(resume, dossier / "resume.json")
    return resume

print("Fonctions prêtes. La cellule suivante entraîne les six modèles.")


## 5. Lancer les six entraînements

Prévoir plusieurs heures sur CPU. Laisser les barres de progression terminer. Les durées affichées peuvent inclure une mise en veille ou une pause du système ; elles ne constituent pas un benchmark matériel.

En cas d’interruption, les sauvegardes des époques complètes permettent de reprendre en réexécutant les cellules de ce même notebook. Les anciens modèles sans augmentation restent dans leurs dossiers d’origine.


In [ ]:
resumes = []
for nom in ARCHITECTURES:
    for graine in GRAINES:
        print(f"\n===== Essai {len(resumes)+1}/6 : {nom}, graine {graine} =====")
        resumes.append(entrainer_avec_augmentation(nom, graine))
        resultats = pd.DataFrame(resumes)
        ecrire_csv(resultats, RACINE / "resultats_augmentation.csv")
        display(resultats[["modele", "graine", "epoque_retenue", "epoque_finale", "val_f1_macro", "val_rappel_pneumonie", "val_specificite_NORMAL"]].round(5))
exiger(len(resultats) == 6, "Certains entraînements sont incomplets.")
print("Les six entraînements sont terminés.")


## 6. Comparer sur la validation originale

Chaque résultat avec augmentation est apparié à l’essai sans augmentation de la même architecture et de la même graine. Un gain positif signifie un score plus élevé avec augmentation. Les moyennes et écarts-types incluent toutes les graines, sans sélectionner celle qui donne le meilleur résultat.

Une seconde synthèse utilise uniquement 123 et 2026, car la graine 42 avait servi à la recherche des hyperparamètres. Les deux synthèses restent exploratoires et utilisent une validation ayant servi à sélectionner les époques.


In [ ]:
METRIQUES = ["val_f1_macro", "val_exactitude", "val_precision_pneumonie", "val_rappel_pneumonie", "val_specificite_NORMAL", "val_f1_pneumonie"]
def comparer_essais(sans_augmentation, avec_augmentation):
    cles = ["modele", "graine"]
    colonnes = cles + ["epoque_retenue"] + METRIQUES
    exiger(not sans_augmentation.duplicated(cles).any() and not avec_augmentation.duplicated(cles).any(), "Essais dupliqués.")
    exiger(set(map(tuple, sans_augmentation[cles].to_numpy())) == set(map(tuple, avec_augmentation[cles].to_numpy())), "Essais non appariés.")
    comparaison = sans_augmentation[colonnes].merge(avec_augmentation[colonnes], on=cles, suffixes=("_sans", "_avec"), validate="one_to_one")
    for metrique in METRIQUES:
        comparaison[f"gain_{metrique}"] = comparaison[f"{metrique}_avec"] - comparaison[f"{metrique}_sans"]
    return comparaison

def synthetiser_comparaison(comparaison):
    colonnes = [f"{m}_{suffixe}" for m in METRIQUES for suffixe in ["sans", "avec"]] + [f"gain_{m}" for m in METRIQUES]
    synthese = comparaison.groupby("modele")[colonnes].agg(["mean", "std"])
    synthese.columns = [f"{mesure}_{stat}" for mesure, stat in synthese.columns]
    return synthese.reset_index()

comparaison = comparer_essais(BASES, resultats)
synthese = synthetiser_comparaison(comparaison)
ecrire_csv(comparaison, RACINE / "comparaison_augmentation_validation.csv")
ecrire_csv(synthese, RACINE / "synthese_augmentation_validation.csv")
ecrire_csv(synthetiser_comparaison(comparaison[comparaison["graine"] != 42]), RACINE / "synthese_augmentation_nouvelles_graines.csv")
display(comparaison[["modele", "graine", "val_f1_macro_sans", "val_f1_macro_avec", "gain_val_f1_macro", "gain_val_rappel_pneumonie", "gain_val_specificite_NORMAL"]].round(5))
display(synthese[["modele", "val_f1_macro_sans_mean", "val_f1_macro_avec_mean", "gain_val_f1_macro_mean", "gain_val_f1_macro_std"]].round(5))

fig, axes = plt.subplots(1, 3, figsize=(13, 4.5))
for axe, metrique, titre in zip(axes, ["val_f1_macro", "val_rappel_pneumonie", "val_specificite_NORMAL"], ["F1 macro", "Rappel PNEUMONIA", "Spécificité NORMAL"]):
    for nom, couleur in [("CNN_classique", "#2563eb"), ("CNN_CBAM", "#c66b14")]:
        sous_tableau = comparaison[comparaison["modele"] == nom].sort_values("graine")
        for numero, (_, ligne) in enumerate(sous_tableau.iterrows()):
            axe.plot([0, 1], [ligne[f"{metrique}_sans"], ligne[f"{metrique}_avec"]], marker="o", color=couleur, alpha=0.65, label=nom if numero == 0 else None)
    axe.set_xticks([0, 1], ["Sans augmentation", "Avec augmentation"])
    axe.set_title(titre)
    axe.set_ylabel("Score sur validation originale")
    axe.grid(axis="y", alpha=0.2)
axes[0].legend(fontsize=9)
fig.suptitle("Chaque ligne relie deux essais de la même architecture et de la même graine")
fig.tight_layout()
fig.savefig(RACINE / "comparaison_augmentation_validation.png", dpi=180, bbox_inches="tight")
plt.show()
plt.close(fig)
ecrire_json({"essais_termines": len(resultats), "graines": GRAINES, "robustesse_reste_a_evaluer": True}, RACINE / "entrainements_termines.json")
print("Fichier principal à transmettre :", RACINE / "resultats_augmentation.csv")
print("Comparaison complémentaire :", RACINE / "comparaison_augmentation_validation.csv")
print("Prochaine mesure : évaluer les sept conditions de robustesse sur ces six nouvelles sauvegardes.")


## Interprétation et suite

Cette expérience ne cherche pas à faire gagner une architecture : elle mesure l’effet d’une augmentation identique sur les deux modèles. Conserver tous les résultats, y compris les baisses. Les hyperparamètres, les limites d’époques et le critère de sélection ont été fixés avant ces six essais.

Les transformations de luminosité et de contraste sont des augmentations courantes ; [la documentation Torchvision](https://docs.pytorch.org/vision/stable/generated/torchvision.transforms.ColorJitter.html) décrit notamment leur tirage aléatoire. Ici, les fonctions sont écrites explicitement pour conserver le padding et séparer les générateurs aléatoires. Elles ne prétendent pas reproduire exactement `ColorJitter`.

La plage [0,90 ; 1,10] est un choix expérimental, pas une plage cliniquement validée. La validation de robustesse suivante reprendra les mêmes transformations fixées dans le notebook 08. Elle comparera à la fois les performances absolues et leur baisse par rapport aux images originales. La calibration des scores et une évaluation indépendante demandent des analyses distinctes.

**Vérifications avant livraison (v2) :** syntaxe de toutes les cellules ; chemins, identifiants d’architecture et configurations alignés sur les notebooks 06 v2, 07 v2 et 08 v2 ; contrôles de références rejoués sur le fichier `resultats_multigraines.csv` v2. L’entraînement réel reste à exécuter avec les images et sauvegardes du projet.
